In [2]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/7aad80ab3fe4ad5ffeabffb7c30bcb70edb218bd/01_data/processed/df_cleaned.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [4]:
# Select relevant features for SA + ML
df_ml = df[
    [
        'lemmatized_tweet',
        'negative',
        'positive',
        'neutral',
        'compound',
        'cyberbullying_type'
    ]
].copy()

# Check shape
print("\n:-------------Shape::-------------", df_ml.shape)

# Check column names
print("\n:-------------Columns::-------------")
print(df_ml.columns.tolist())

# Check missing values
print("\n----------------Missing values:-------------")
print(df_ml.isnull().sum())

# Display first 5 rows
df_ml.head()


:-------------Shape::------------- (42645, 6)

:-------------Columns::-------------
['lemmatized_tweet', 'negative', 'positive', 'neutral', 'compound', 'cyberbullying_type']

----------------Missing values:-------------
lemmatized_tweet      30
negative               0
positive               0
neutral                0
compound               0
cyberbullying_type     0
dtype: int64


,lemmatized_tweet,negative,positive,neutral,compound,cyberbullying_type
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000,not_cyberbullying
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137,not_cyberbullying
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854,not_cyberbullying
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258,not_cyberbullying


In [5]:
PROFANITY_WORDS = {
    'fuck', 'fucking', 'fucked', 'fuckin',
    'shit', 'shitty',
    'bitch', 'bitches',
    'asshole', 'assholes',
    'bastard',
    'damn',
    'dick',
    'piss',
    'crap',
    'bullshit',
    'motherfucker',
    'idiot',
    'stupid',
    'moron',
    'retard',
    'retarded'
}

In [6]:
def extract_linguistic_features(text):

    text = str(text)

    # Basic tokens
    words = re.findall(r'\b\w+\b', text.lower())

    word_count = len(words)
    char_count = len(text)
    unique_words = len(set(words))

    # Profanity
    profanity_words = [
        word for word in words
        if word in PROFANITY_WORDS
    ]

    profanity_count = len(profanity_words)
    unique_profanity_count = len(set(profanity_words))

    profanity_ratio = (
        profanity_count / word_count
        if word_count > 0 else 0
    )

    profanity_presence = int(profanity_count > 0)

    # Second-person language
    second_person_words = {
        'you',
        'your',
        'yours',
        'yourself',
        'yourselves',
        'youre',
        'youve',
        'youll',
        'youd'
    }

    second_person_count = sum(
        word in second_person_words
        for word in words
    )

    # Punctuation
    exclamation_count = text.count('!')
    question_count = text.count('?')
    comma_count = text.count(',')
    period_count = text.count('.')

    repeated_exclamation_count = len(
        re.findall(r'!{2,}', text)
    )

    repeated_question_count = len(
        re.findall(r'\?{2,}', text)
    )

    repeated_punctuation_count = len(
        re.findall(r'[!?]{2,}', text)
    )

    # Mentions / Hashtags
    mention_count = len(
        re.findall(r'@\w+', text)
    )

    hashtag_count = len(
        re.findall(r'#\w+', text)
    )

    # Return features
    return pd.Series({
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_count': unique_words,

        'profanity_count': profanity_count,
        'unique_profanity_count': unique_profanity_count,
        'profanity_ratio': profanity_ratio,
        'profanity_presence': profanity_presence,

        'second_person_count': second_person_count,

        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'comma_count': comma_count,
        'period_count': period_count,

        'repeated_exclamation_count': repeated_exclamation_count,
        'repeated_question_count': repeated_question_count,
        'repeated_punctuation_count': repeated_punctuation_count,

        'mention_count': mention_count,
        'hashtag_count': hashtag_count
    })

In [7]:
import re
linguistic_features = df['tweet_text'].apply(
    extract_linguistic_features
)

df = pd.concat(
    [df, linguistic_features],
    axis=1
)

In [8]:
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [9]:
# ============================================================
# CELL 1 — SETUP + DATA PREPARATION (uses your existing `df`)
# ============================================================
import time, warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.sparse import hstack, csr_matrix
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
N_SPLITS     = 5        # 5-fold CV (every row is predicted exactly once)
RANDOM_STATE = 42
TEXT_COL     = "lemmatized_tweet"
RAW_TEXT_COL = "tweet_text"          # only used to show the tweet in the error sheet
TARGET_COL   = "cyberbullying_type"
NUMERIC_FEATURES = [
    "negative", "positive", "neutral", "compound",
    "second_person_count", "mention_count", "hashtag_count",
    "profanity_count", "unique_profanity_count",
    "profanity_ratio", "profanity_presence"
]
FOCUS_CLASSES = ["not_cyberbullying", "other_cyberbullying"]

# ---------------- DATA ----------------
df_cv = df[[RAW_TEXT_COL, TEXT_COL] + NUMERIC_FEATURES + [TARGET_COL]].copy()
df_cv = df_cv.dropna(subset=[TARGET_COL])          # original df index is kept as row_id

X_text = df_cv[TEXT_COL].fillna("").astype(str).to_numpy()
X_num = (
    df_cv[NUMERIC_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .to_numpy(dtype=float)
)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_cv[TARGET_COL].astype(str))

class_names  = [str(c) for c in label_encoder.classes_]
class_ids    = np.arange(len(class_names))
class_to_id  = {c: i for i, c in enumerate(class_names)}

print("Total rows used for CV :", len(y))
print("Classes                :", class_names)
print("Missing text rows      :", int((df_cv[TEXT_COL].isna()).sum()), "(filled with empty string)")

Total rows used for CV : 42645
Classes                : ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']
Missing text rows      : 30 (filled with empty string)


In [10]:
# ============================================================
# CELL 2 — BUILD STRATIFIED K-FOLD FEATURE MATRICES (ONCE)
# TF-IDF is fitted ONLY on each fold's training part (no leakage)
# and cached, so all 7 models reuse the same matrices.
# ============================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_data = []
for fold, (tr_idx, va_idx) in enumerate(
    tqdm(skf.split(X_text, y), total=N_SPLITS, desc="Building folds"), start=1
):
    tfidf = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2),
        max_features=10000, sublinear_tf=True
    )
    W_tr = tfidf.fit_transform(X_text[tr_idx])
    W_va = tfidf.transform(X_text[va_idx])

    fold_data.append({
        "fold"  : fold,
        "va_idx": va_idx,
        "y_tr"  : y[tr_idx],
        "y_va"  : y[va_idx],
        "X_tr"  : hstack([W_tr, csr_matrix(X_num[tr_idx])], format="csr"),
        "X_va"  : hstack([W_va, csr_matrix(X_num[va_idx])], format="csr"),
    })
    print(f"Fold {fold}: train={len(tr_idx)}, val={len(va_idx)}, "
          f"features={fold_data[-1]['X_tr'].shape[1]}")

Building folds:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 1: train=34116, val=8529, features=10011
Fold 2: train=34116, val=8529, features=10011
Fold 3: train=34116, val=8529, features=10011
Fold 4: train=34116, val=8529, features=10011
Fold 5: train=34116, val=8529, features=10011


In [11]:
# ============================================================
# CELL 3 — 7 MODELS WITH BEST HYPERPARAMETERS (no grid search)
# LR / Linear SVM  : settings from your baseline cell
# Other 5 models   : best params found in your Experiment 2 grid search
# ============================================================
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=10000,
        C=1.0, solver="lbfgs", random_state=42
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced", max_iter=10000,
        C=0.1, random_state=42
    ),

    "XGBoost": XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        tree_method="hist", random_state=42, n_jobs=-1,
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8
    ),

    "SGD Classifier": SGDClassifier(
        class_weight="balanced", max_iter=10000, tol=1e-3,
        random_state=42,
        loss="hinge", alpha=1e-4, penalty="l2"
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1,
        n_estimators=200, max_depth=40, min_samples_leaf=1
    ),

    "Extra Trees": ExtraTreesClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1,
        n_estimators=200, max_depth=40, min_samples_leaf=1
    ),

    "MLP Neural Network": Pipeline([
        ("scaler", MaxAbsScaler()),
        ("mlp", MLPClassifier(
            solver="adam", activation="relu", batch_size=128,
            max_iter=100, early_stopping=True, n_iter_no_change=8,
            random_state=42,
            hidden_layer_sizes=(128,), alpha=1e-3
        ))
    ]),
}

model_names = list(models.keys())
print("Models:", model_names)

Models: ['Logistic Regression', 'Linear SVM', 'XGBoost', 'SGD Classifier', 'Random Forest', 'Extra Trees', 'MLP Neural Network']


In [12]:
# ============================================================
# CELL 4 — K-FOLD OUT-OF-FOLD (OOF) PREDICTIONS FOR ALL 7 MODELS
# Every row of the dataset gets one prediction per model, made by a
# model that never saw that row. Finished models are skipped on
# re-run; to recompute one: del oof_pred["XGBoost"]
# ============================================================
if "oof_pred" not in globals():
    oof_pred = {}          # model -> array of predicted class ids (len = all rows)
    fold_macro_f1 = {}     # model -> list of per-fold macro-F1
    run_minutes = {}       # model -> total minutes

for name, template in models.items():

    if name in oof_pred:
        print(f"[skip] {name} already done")
        continue

    preds = np.full(len(y), -1, dtype=int)
    fold_scores = []
    t0 = time.time()

    for fd in tqdm(fold_data, desc=name, leave=False):
        model = clone(template)
        model.fit(fd["X_tr"], fd["y_tr"])
        p = np.asarray(model.predict(fd["X_va"])).astype(int).ravel()

        preds[fd["va_idx"]] = p
        fold_scores.append(f1_score(fd["y_va"], p, average="macro", zero_division=0))

    assert (preds >= 0).all(), "Some rows did not get an OOF prediction"

    oof_pred[name] = preds
    fold_macro_f1[name] = fold_scores
    run_minutes[name] = (time.time() - t0) / 60

    print(f"{name:22s} | OOF Macro-F1 = "
          f"{f1_score(y, preds, average='macro'):.4f} | "
          f"folds mean±std = {np.mean(fold_scores):.4f}±{np.std(fold_scores):.4f} | "
          f"{run_minutes[name]:.2f} min")

Logistic Regression:   0%|          | 0/5 [00:00<?, ?it/s]

Logistic Regression    | OOF Macro-F1 = 0.8293 | folds mean±std = 0.8293±0.0028 | 1.62 min


Linear SVM:   0%|          | 0/5 [00:00<?, ?it/s]

Linear SVM             | OOF Macro-F1 = 0.8284 | folds mean±std = 0.8284±0.0036 | 0.24 min


XGBoost:   0%|          | 0/5 [00:00<?, ?it/s]

...

Random Forest          | OOF Macro-F1 = 0.8235 | folds mean±std = 0.8234±0.0044 | 0.94 min


Extra Trees:   0%|          | 0/5 [00:00<?, ?it/s]

Extra Trees            | OOF Macro-F1 = 0.7930 | folds mean±std = 0.7928±0.0077 | 0.50 min


MLP Neural Network:   0%|          | 0/5 [00:00<?, ?it/s]

MLP Neural Network     | OOF Macro-F1 = 0.8207 | folds mean±std = 0.8207±0.0020 | 8.29 min


In [13]:
# ============================================================
# CELL 5 — CROSS-VALIDATED PERFORMANCE (ALL DATA, OOF)
# ============================================================
nc_id = class_to_id["not_cyberbullying"]
oc_id = class_to_id["other_cyberbullying"]

rows = []
for name in model_names:
    p = oof_pred[name]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y, p, labels=class_ids, zero_division=0
    )
    rows.append({
        "Model"              : name,
        "OOF Accuracy"       : accuracy_score(y, p),
        "OOF Macro-F1"       : f1_score(y, p, average="macro", zero_division=0),
        "OOF Weighted-F1"    : f1_score(y, p, average="weighted", zero_division=0),
        "Fold Macro-F1 mean" : np.mean(fold_macro_f1[name]),
        "Fold Macro-F1 std"  : np.std(fold_macro_f1[name]),
        "Not-CB Precision"   : prec[nc_id],
        "Not-CB Recall"      : rec[nc_id],
        "Not-CB F1"          : f1[nc_id],
        "Other-CB Precision" : prec[oc_id],
        "Other-CB Recall"    : rec[oc_id],
        "Other-CB F1"        : f1[oc_id],
        "Runtime (min)"      : run_minutes.get(name, np.nan),
    })

cv_results_df = (
    pd.DataFrame(rows)
    .sort_values("OOF Macro-F1", ascending=False)
    .reset_index(drop=True)
)

print(f"{N_SPLITS}-FOLD CROSS-VALIDATION RESULTS (all {len(y)} rows)")
display(cv_results_df.round(4))

5-FOLD CROSS-VALIDATION RESULTS (all 42645 rows)


,Model,OOF Accuracy,OOF Macro-F1,OOF Weighted-F1,Fold Macro-F1 mean,Fold Macro-F1 std,Not-CB Precision,Not-CB Recall,Not-CB F1,Other-CB Precision,Other-CB Recall,Other-CB F1,Runtime (min)
0,XGBoost,0.8533,0.8362,0.8565,0.8362,0.0033,0.5755,0.6850,0.6255,0.5994,0.5719,0.5853,11.5775
1,Logistic Regression,0.8454,0.8293,0.8484,0.8293,0.0028,0.5934,0.5913,0.5924,0.5675,0.6711,0.6150,1.6197
2,Linear SVM,0.8467,0.8284,0.8476,0.8284,0.0036,0.6040,0.5718,0.5874,0.5754,0.6607,0.6151,0.2413
3,SGD Classifier,0.8435,0.8241,0.8437,0.8239,0.0036,0.5770,0.6002,0.5883,0.6024,0.5980,0.6002,0.0820
4,Random Forest,0.8384,0.8235,0.8437,0.8234,0.0044,0.5673,0.6222,0.5935,0.5401,0.6441,0.5875,0.9384
5,MLP Neural Network,0.8401,0.8207,0.8409,0.8207,0.0020,0.5777,0.6095,0.5932,0.5844,0.5773,0.5808,8.2906
6,Extra Trees,0.8200,0.7930,0.8181,0.7928,0.0077,0.5110,0.7792,0.6172,0.5945,0.3509,0.4413,0.4985


In [14]:
# ============================================================
# CELL 6 — ERROR SHEET (only rows whose TRUE class is
#          not_cyberbullying or other_cyberbullying)
#
# error_count : in how many of the 7 models this row was misclassified (0-7)
# error_type  : "true_label -> wrong label predicted most often among the
#               models that failed on this row" (tie -> first model in list)
# ============================================================
focus_ids  = [class_to_id[c] for c in FOCUS_CLASSES]
focus_mask = np.isin(y, focus_ids)

sheet = pd.DataFrame({
    "row_id"    : df_cv.index.to_numpy(),      # index of the row in your original df
    "tweet_text": df_cv[RAW_TEXT_COL].to_numpy(),
    "true_label": label_encoder.inverse_transform(y),
})

for name in model_names:
    sheet[f"pred_{name}"]  = label_encoder.inverse_transform(oof_pred[name])
    sheet[f"wrong_{name}"] = (oof_pred[name] != y).astype(int)

wrong_cols = [f"wrong_{n}" for n in model_names]
sheet["error_count"] = sheet[wrong_cols].sum(axis=1)

focus_sheet = sheet[focus_mask].copy()

def get_error_type(row):
    if row["error_count"] == 0:
        return "no_error"
    wrong_preds = [
        row[f"pred_{n}"] for n in model_names
        if row[f"pred_{n}"] != row["true_label"]
    ]
    dominant = Counter(wrong_preds).most_common(1)[0][0]
    return f"{row['true_label']} -> {dominant}"

def get_models_wrong(row):
    return ", ".join(n for n in model_names if row[f"wrong_{n}"] == 1)

focus_sheet["error_type"]   = focus_sheet.apply(get_error_type, axis=1)
focus_sheet["models_wrong"] = focus_sheet.apply(get_models_wrong, axis=1)

# Column order: the important columns first
front = ["row_id", "tweet_text", "true_label", "error_type", "error_count", "models_wrong"]
rest  = [c for c in focus_sheet.columns if c not in front]
focus_sheet = focus_sheet[front + rest].reset_index(drop=True)

# Error sheet = only rows misclassified by at least one model
error_sheet = (
    focus_sheet[focus_sheet["error_count"] > 0]
    .sort_values(["error_count", "true_label"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Rows in the two focus classes :", len(focus_sheet))
print("Rows with >= 1 model wrong    :", len(error_sheet))
print("Rows wrong in ALL 7 models    :", int((error_sheet['error_count'] == len(model_names)).sum()))

print("\nError-count distribution (focus classes):")
display(
    focus_sheet.groupby(["true_label", "error_count"]).size()
    .unstack(fill_value=0)
)

display(error_sheet[["row_id", "tweet_text", "true_label",
                     "error_type", "error_count", "models_wrong"]].head(15))

Rows in the two focus classes : 11806
Rows with >= 1 model wrong    : 8120
Rows wrong in ALL 7 models    : 1516

Error-count distribution (focus classes):


error_count,0,1,2,3,4,5,6,7
true_label,,,,,,,,
not_cyberbullying,2497,695,401,418,573,541,612,708
other_cyberbullying,1189,947,641,566,388,381,441,808


,row_id,tweet_text,true_label,error_type,error_count,models_wrong
0,3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
1,4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,not_cyberbullying -> religion,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
2,9,the bully flushes on kd http://twitvid.com/a2tnp,not_cyberbullying,not_cyberbullying -> age,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
3,28,"the thing about uber is, even if it wasn't a h...",not_cyberbullying,not_cyberbullying -> gender,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
4,36,fuck this paper bullshit,not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
5,41,if you are feeling perturbed about women that ...,not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
6,42,you look like a gypsy. sorry i'm not sorry.,not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
7,44,"@hayhenning good news, if i ever have a proble...",not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
8,50,lil bro @mackmanee_15 was on the bully last night,not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."
9,54,"@carsalesbossman unlike men, who can't fix car...",not_cyberbullying,not_cyberbullying -> other_cyberbullying,7,"Logistic Regression, Linear SVM, XGBoost, SGD ..."


In [15]:
# ============================================================
# CELL 7 — SUMMARY TABLES + SAVE EXCEL / CSV
# ============================================================
# (a) error type x error count
summary_type = pd.crosstab(error_sheet["error_type"], error_sheet["error_count"])
summary_type["Total"] = summary_type.sum(axis=1)
summary_type = summary_type.sort_values("Total", ascending=False)

# (b) per-model errors inside the two focus classes
pm_rows = []
for name in model_names:
    for cls in FOCUS_CLASSES:
        m = focus_sheet["true_label"] == cls
        n_wrong = int(focus_sheet.loc[m, f"wrong_{name}"].sum())
        pm_rows.append({
            "Model"      : name,
            "True class" : cls,
            "Samples"    : int(m.sum()),
            "Errors"     : n_wrong,
            "Error rate" : n_wrong / m.sum()
        })
per_model_errors = pd.DataFrame(pm_rows)

print("ERROR TYPE x ERROR COUNT")
display(summary_type)
print("\nPER-MODEL ERRORS (focus classes)")
display(per_model_errors.round(4))

# (c) save
OUT_XLSX = "error_sheet_kfold.xlsx"
OUT_CSV  = "error_sheet_kfold.csv"

error_sheet.to_csv(OUT_CSV, index=False)

try:
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        error_sheet.to_excel(writer,      sheet_name="Error_Sheet",      index=False)
        focus_sheet.to_excel(writer,      sheet_name="All_Focus_Rows",   index=False)
        summary_type.to_excel(writer,     sheet_name="ErrorType_Summary")
        per_model_errors.to_excel(writer, sheet_name="Per_Model_Errors", index=False)
        cv_results_df.to_excel(writer,    sheet_name="CV_Results",       index=False)
    print(f"\nSaved: {OUT_XLSX}  and  {OUT_CSV}")
except ImportError:
    print(f"\nopenpyxl not found -> saved only {OUT_CSV}  (pip install openpyxl for Excel)")

# Colab users: uncomment to download
# from google.colab import files
# files.download(OUT_XLSX)

ERROR TYPE x ERROR COUNT


error_count,1,2,3,4,5,6,7,Total
error_type,,,,,,,,
other_cyberbullying -> not_cyberbullying,678,514,473,316,277,358,691,3307
not_cyberbullying -> other_cyberbullying,471,289,298,425,405,472,382,2742
not_cyberbullying -> gender,118,38,31,40,39,33,91,390
not_cyberbullying -> age,42,44,49,60,37,57,70,359
not_cyberbullying -> religion,43,21,28,31,42,43,141,349
other_cyberbullying -> gender,53,35,37,32,48,49,70,324
other_cyberbullying -> ethnicity,113,31,17,14,17,13,12,217
other_cyberbullying -> age,53,31,26,14,17,13,29,183
other_cyberbullying -> religion,50,30,13,12,22,8,6,141



PER-MODEL ERRORS (focus classes)


,Model,True class,Samples,Errors,Error rate
0,Logistic Regression,not_cyberbullying,6445,2634,0.4087
1,Logistic Regression,other_cyberbullying,5361,1763,0.3289
2,Linear SVM,not_cyberbullying,6445,2760,0.4282
3,Linear SVM,other_cyberbullying,5361,1819,0.3393
4,XGBoost,not_cyberbullying,6445,2030,0.3150
5,XGBoost,other_cyberbullying,5361,2295,0.4281
6,SGD Classifier,not_cyberbullying,6445,2577,0.3998
7,SGD Classifier,other_cyberbullying,5361,2155,0.4020
8,Random Forest,not_cyberbullying,6445,2435,0.3778
9,Random Forest,other_cyberbullying,5361,1908,0.3559



Saved: error_sheet_kfold.xlsx  and  error_sheet_kfold.csv


In [16]:
# import pandas as pd

# # Assuming 'df' is your existing Pandas DataFrame

# # 1. Save as a CSV file
# error_sheet.to_csv("error_sheet.csv", index=False)  # index=False prevents writing row numbers

In [1]:
# ============================================================
# CELL 8 — WHO MISCLASSIFIES "not_cyberbullying" ROWS?
# Uses: focus_sheet, model_names (from Cell 6)
# ============================================================
NOT_CB = "not_cyberbullying"
wrong_cols = [f"wrong_{n}" for n in model_names]

nc = focus_sheet[focus_sheet["true_label"] == NOT_CB].copy()
n_total    = len(nc)
affected   = nc[nc["error_count"] > 0]          # rows at least one model got wrong
n_affected = len(affected)

# ---------- (A) AVERAGE AFFECTED ROWS ----------
errors_per_model = nc[wrong_cols].sum()
errors_per_model.index = model_names

print("=" * 70)
print("A) AVERAGE AFFECTED ROWS  (true class = not_cyberbullying)")
print("=" * 70)
print(f"Total not_cyberbullying rows                    : {n_total}")
print(f"Rows wrongly classified by >=1 model (affected) : {n_affected} "
      f"({n_affected / n_total:.2%})")
print(f"Rows wrongly classified by ALL {len(model_names)} models       : "
      f"{int((nc['error_count'] == len(model_names)).sum())}")
print(f"Avg rows misclassified per model                : "
      f"{errors_per_model.mean():.1f} ({errors_per_model.mean() / n_total:.2%})")
print(f"Avg error_count per not_cyberbullying row       : {nc['error_count'].mean():.3f}")
print(f"Avg error_count per AFFECTED row (of 7 models)  : {affected['error_count'].mean():.3f}")

print("\nHow many models fail on each row (error_count distribution):")
dist = nc["error_count"].value_counts().sort_index().rename("rows").to_frame()
dist["% of not_cb rows"] = (dist["rows"] / n_total * 100).round(2)
display(dist)

# ---------- (B) WHICH MODELS ARE WRONG ----------
who = pd.DataFrame({
    "Errors"        : errors_per_model,
    "Error rate"    : errors_per_model / n_total,
    "Share of affected rows": errors_per_model / n_affected,
    # only this model failed on the row
    "Exclusive errors (only this model wrong)": [
        int(((nc[f"wrong_{n}"] == 1) & (nc["error_count"] == 1)).sum())
        for n in model_names
    ],
    # this model failed and the majority (>=4 of 7) also failed
    "Errors on majority-wrong rows (>=4 models)": [
        int(((nc[f"wrong_{n}"] == 1) & (nc["error_count"] >= 4)).sum())
        for n in model_names
    ],
})
who["Above avg?"] = np.where(who["Errors"] > errors_per_model.mean(), "yes", "no")
who = who.sort_values("Errors", ascending=False)

print("\n" + "=" * 70)
print("B) WHICH MODELS MISCLASSIFY not_cyberbullying")
print("=" * 70)
display(who.round(4))

# ---------- (C) WHERE DO THE WRONG ROWS GO? ----------
dest = {}
for n in model_names:
    wrong_rows = nc[nc[f"wrong_{n}"] == 1]
    dest[n] = wrong_rows[f"pred_{n}"].value_counts()
dest_df = pd.DataFrame(dest).fillna(0).astype(int).T
dest_df["Total wrong"] = dest_df.sum(axis=1)
dest_df = dest_df.loc[who.index]                     # same order as table B

print("\n" + "=" * 70)
print("C) not_cyberbullying rows were wrongly predicted AS (per model)")
print("=" * 70)
display(dest_df)

dest_pct = dest_df.drop(columns="Total wrong").div(dest_df["Total wrong"], axis=0) * 100
print("\nSame, in % of each model's errors:")
display(dest_pct.round(1))

# ---------- (D) HARDEST ROWS (most models fail) ----------
hard = (
    nc[nc["error_count"] >= 4]
    .sort_values("error_count", ascending=False)
    [["row_id", "tweet_text", "error_type", "error_count", "models_wrong"]]
)
print("\n" + "=" * 70)
print(f"D) not_cyberbullying rows wrong in >=4 of {len(model_names)} models: {len(hard)}")
print("=" * 70)
display(hard.head(20))

# ---------- SAVE ----------
with pd.ExcelWriter("not_cb_error_analysis.xlsx", engine="openpyxl") as w:
    dist.to_excel(w,     sheet_name="ErrorCount_Dist")
    who.to_excel(w,      sheet_name="Who_Misclassifies")
    dest_df.to_excel(w,  sheet_name="Predicted_As")
    dest_pct.round(1).to_excel(w, sheet_name="Predicted_As_pct")
    hard.to_excel(w,     sheet_name="Hard_Rows", index=False)
print("\nSaved: not_cb_error_analysis.xlsx")

NameError: name 'model_names' is not defined